In [ ]:
# Paths
train_path = "../dataset/output/train_splited"
test_path = "../dataset/output/test_splited"
valid_path = "../dataset/output/valid_splited"

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Parameters
batch_size = 32
image_size = (224, 224)

# Data Generators
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

valid_generator = datagen.flow_from_directory(
    valid_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

test_generator = datagen.flow_from_directory(
    test_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import DenseNet169
from tensorflow.keras.callbacks import ModelCheckpoint

# Load the DenseNet model
base_model = DenseNet169(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add new layers to the model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(7, activation='softmax')(x)

# Define the new model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# Model Checkpoint
checkpoint_filepath = "models/densenet_multiclass-{epoch:02d}-{val_accuracy:.2f}.hdf5"
checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True,
    mode='max'
)

In [ ]:
# Train the model
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=valid_generator,
    callbacks=[checkpoint]
)

In [ ]:
from tensorflow.keras.models import load_model

# Load the best model based on validation accuracy
# model = load_model("models/densenet_multiclass-xx-xx.hdf5")  # Replace with the actual best model filename